# Training
The training loop generally goes as follows:
1. Get batch of text
2. Tokenize
3. Forward pass (emebd -> transformer layers -> predict0
4. Compute loss
5. Backward pass
6. Update weights

We have already been over/implemented steps 2 through 4 (step 1 is just loading the data and shaping it correctly).

## Backpropagation
We mentioned the concept of backpropigation before in the context of why normalization is necessary. In short, through the use of chain rule, we can compute how much each parameter contributed to the loss. Consider a simple example:

$L = x, x = 3y, y = \frac{z}{2}$

To compute $\frac{∂L}{∂z}$, we need the chain rule:

$\frac{∂L}{∂z}= \frac{∂L}{∂x}*\frac{∂x}{∂y}*\frac{∂y}{∂z} = 1*3*\frac{1}{2}=\frac{3}{2}$

Note that I used a very simple example but say $L=x^2$, then $\frac{∂L}{∂x} = 2x$ so $\frac{∂L}{∂z}$ would depend on $x$. So some gradients depend on forward pass values- for this reason PyTorch saves tehm during the forward pass.

### Gradient Descent
Now that we know how much each parameter contributes to the loss via its gradient, we need to think about how we want to adjust our parameters:
```weight = weight + update```. But do we just want an update to be one full step of the gradient? Probably not, as that be a massive overcorrection in our training, especially as loss can be as low as <1. So an improvement would be to introdue a learning rate: ```update = -learning_rate * gradient```. But simple gradient descent has two main problems:
1. The learning rate is the same for every parameter (some need bigger steps and others need smaller)
2. Gradients are noisy batch-to-batch, so we can oscillate (a nod to the overstep issue above)

### The Solution: AdamW
Adaptive Moment estimation with Weight decay fixes the problems that we encounter with a simple learning rate:

1. Adaptive learning rate: at a high level, this gives each parameter its own learning rate that depends on how much it has been moving. More granularly, it tracks the average of squared gradients per parameter: if a parameter's gradients have been larged historically, its effective learning rate shrinks. If they have been small, it gets a larger step. So more sensitive parameters get smaller updates, and less sensitive parameters get larger updates.

    $V_{i+1} = \frac{𝝱_2 * V_i + (1-β_2)*grad}{1-β_2^t}$


2. Momentum: smooths out noisy gradients: instead of using the raw gradient, it uses a running avreage of past gradients. So if gradients keep pointing in the same direction, momentum builds and we move faster. but if they are noisy and fickle, they cancel out and we move slower.

    $M_{i+1} = \frac{𝝱_1 * M_i + (1-β_1)*grad}{1-β_1^t}$

3. Weight decay: Not included in traditional Adam (hence the 'W'), decoupled (separate from gradient update so as to not affect adaptive learning rates) weight decay adds a direct penalty that shrinks every weight toward zero each step, independent of the gradient. This prevents weights from growing arbitrarily large. Large weights mean the model is very sensitive to small input changes, which is a sign of overfitting. Weight decay prevents the model from "learning" specific patters rather than general ones.

$w = w(1-lr*w_{decay}) \rightarrow w = w - lr*\frac{M_{i+1}}{\sqrt{V_{i+1}}+ϵ}$

Note that we do not shrink every weight; we split the parameters into decay and no-decay groups (biases and norm layer weights are excluded). Taking a step back: weight decay pulls weight toward zero every step, independent of gradient. This works for a linear layer's weight matrix since we do not want these weights to be getting too large. But RMSNorm's gamma starts at 1.0 and learns to scale activations. So does the biases. If we shrink them every step, we fight against what they are trying to learn.

## Learning Rate Scheduler
Early in training, gradients are noisy and weights are random - a big LR causes chaos. Later in training, we want smaller steps to fine tune. We handle both via a scheduler.

1. **Warmup**: LR starts near zero and increases linearly for the first N steps, going from ~0 to ```max_lr``` over ```warmup_steps```. This prevents unstable updates at the start when weights are random: ```max_lr * t / warmup_steps```
2. **Cosine Decay**: LR gradually decreases following a cosine curve down to a minimum.

$$LR_t = LR_{min} + (LR_{max} - LR_{min}) \cdot \frac{1 + \cos\left(\frac{\pi(t - t_{warmup})}{t_{max} - t_{warmup}}\right)}{2}$$

3. **Floor**: stays at a minimum learning rate; never hits zero. We never really reach this point unless we go past the maximum training step, so it mostly exists as a fallback.

## Putting it together
AdamW handles per-parameter adaptive learning rates internally. The scheduler sets the global learning rate that AdamW scales from.

## Quick note: zero_grad and gradient accumulation
PyTorch accumulates gradients by default-- every time we call ```loss.backward()```, it adds to the existing ```.grad``` values rather than replacing them. If we do not zero them out, gradients from the previous batch contaminate the current batch's updates.

**Gradient accumulation**: when our GPU cannot fit a large batch, we do several forward passes letting the gradients accumulate. For instance, say we want effective ```batch_size = 48``` but our GPU only fits 8 sequences at once. Without accumulation we would need ```batch-48``` in one forward pass which would cause a crash. With accumulation:
```
forward(batch_1_of_6) → loss/6 → backward()  # gradients accumulate
forward(batch_2_of_6) → loss/6 → backward()  # gradients accumulate  
forward(batch_3_of_6) → loss/6 → backward()  # gradients accumulate
forward(batch_4_of_6) → loss/6 → backward()  # gradients accumulate
optimizer.step()                              # one update using all 6 batches
optimizer.zero_grad()                         # now reset
```

In [5]:
import torch
from torch.utils.data import Dataset
import time
import os
from datasets import load_dataset
import matplotlib as plt

In [1]:
"""
Four things to instantiate:
1. Dataset / Data Loader
2. Model
3. Optimizer
4. Scheduler

==== Data Set ====
"""

class TextDataset(Dataset):
  def __init__(self, texts:list[str], tokenizer, max_seq_len: int = 2048):
    self.tokenizer = tokenizer
    self.max_seq_len = max_seq_len

    full_tokens = []

    for text in texts:
      tokens = tokenizer.encode(text)
      full_tokens.extend(tokens)
      full_tokens.append(tokenizer.eos_token_id)

    self.full_tokens = torch.tensor(full_tokens, dtype=torch.long)
    print(f"Total tokens: {len(self.full_tokens)}")

  def __len__(self):
    return(len(self.full_tokens) - 1) // self.max_seq_len


  def __getitem__(self, idx: int) -> tuple:
    start = idx * self.max_seq_len
    end = start + self.max_seq_len

    input_ids = self.full_tokens[start:end]
    target_ids = self.full_tokens[start:end]
    target_ids = self.tokens[start + 1: end + 1]

    return input_ids, target_ids

In [2]:
# Loading Data
# configurable
def load_training_data(max_samples: int = 10000):
  dataset = load_dataset("HuggingFaceFW/fineweb-edu", split="train", streaming=True)
  return [item["text"] for item in dataset.take(max_samples)]

In [ ]:
# Optimizer
def create_optimizer(model, config):
  to_decay = []
  no_decay = []
  # A parameter should NOT get weight decay if it is 1D (biases, norm gains) or name has "norm" or "bias"
  for name, param in model.named_parameters():
    if not param.requires_grad:
      continue
    if param.dim() <= 1 or "norm" in name.lower() or "bias" in name.lower():
      no_decay.append(param)
    else:
      to_decay.append(param)

  adamw = torch.optim.AdamW([
      {"params": to_decay, "weight_decay": config.weight_decay},
      {"params": no_decay, "weight_decay": 0.0}
  ], lr = config.learning_rate, betas = (config.beta1, config.beta2), eps = config.epsilon)

  return adamw


In [3]:
import math

# configurable
class CosineWarmupScheduler:
  def __init__(self, optimizer, warmup_steps, max_steps, max_lr = 3e-4, min_lr = 1e-5):
    self.optimizer = optimizer
    self.warmup_steps = warmup_steps
    self.max_steps = max_steps
    self.max_lr = max_lr
    self.min_lr = min_lr
    self.current_step = 0

  def get_lr(self):
    # Warmup
    if self.current_step < self.warmup_steps:
      return self.max_lr * self.current_step / self.warmup_steps
    # cosine stage
    elif self.current_step < self.max_steps:
      progress = (self.current_step - self.warmup_steps) / (self.max_steps - self.warmup_steps)
      cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
      return self.min_lr + (self.max_lr - self.min_lr) * cosine_decay
    # we return the minimum LR when training goes past the scheduled max: this is just a fallback
    return self.min_lr

  def step(self):
    lr = self.get_lr()
    for param_group in self.optimizer.param_groups:
      param_group["lr"] = lr
    self.current_step += 1

  def state_dict(self):
    return{"current_step": self.current_step}

  def load_state_dict(self, state_dict):
    self.current_step = state_dict["current_step"]

In [ ]:
def train(model, train_dataset, config, device: None, save_dir = "checkpoints"):
  os.makedirs(save_dir, exist_ok = True)
  model = model.to(device)
  model.train()

  # 1. Batch
  dataloader = torch.utils.data.DataLoader(train_dataset, batch_size = config.batch_size, shuffle = True, drop_last = True)

  # 2. Optimizer
  optimizer = create_optimizer(model, config)

  # 3. Scheduler
  scheduler = CosineWarmupScheduler(optimizer, warmup_steps=config.warmup_steps,
                                    max_steps = config.max_steps, max_lr =config.max_lr,
                                    min_lr = config.min_lr)
  """
  use_amp = device.type == "cuda"
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp) if use_amp else None
  """
  step_count = 0
  total_loss = 0
  loss_hist = []
  while step_count < config.max_steps:
    # loop through all the batches
    for input_ids, target_ids in dataloader:
      # moves a tensor from a CPU to a GPU (wherever device points)
      input_ids = input_ids.to(device)
      target_ids = target_ids.to(device)
      # zero grad
      optimizer.zero_grad()
      # forward pass and loss
      logits, loss = model.forward(input_ids, target_ids)

      # backward pass
      loss.backward()

      # gradient clipping: caps magnitude of gradients before weight update, configurable
      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 1.)

      # update weights
      optimizer.step()
      scheduler.step()

      # logging, configurable
      total_loss += loss.item()
      loss_hist.append((step_count, loss.item()))
      if step_count % 100 == 0:
        print(f"Step count: {step_count} | Loss: {loss.item()}")

      step_count += 1
  return loss_hist


def plot_loss(loss_hist):
  plt.figure(figsize=(10, 5))
  steps, losses = zip(*loss_hist)
  plt.plot(steps, losses)
  plt.xlabel("Step")
  plt.ylabel("Loss")
  plt.title("Training Loss")